# Gold — Índice-proxy e indicadores de mercado

Lê a tabela `poc_b3_modernizacao.silver.cotacoes` (dados limpos, sem os registros em
quarentena) e calcula os indicadores de negócio do projeto.

**Camada gerada exclusivamente de forma automatizada** — sem edição manual em nenhuma
hipótese, especificamente para eliminar divergência de números entre diferentes
consumidores do dado (ver seção de governança em `docs/architecture.md`).

Indicadores calculados:
1. **Retorno diário por ticker**: `(preço atual - fechamento anterior) / fechamento anterior`
2. **Índice-proxy**: média simples do retorno diário dos 4 papéis
3. **Ranking de valorização**: ordena os papéis do maior para o menor retorno do dia
4. **Dispersão do dia**: desvio padrão dos 4 retornos diários

*Indicador futuro (pendente de pelo menos 2 dias de dados válidos): variação acumulada
do índice entre dias.*

**Entrada:** tabela `poc_b3_modernizacao.silver.cotacoes`
**Saída:** tabela `poc_b3_modernizacao.gold.indicadores_diarios` (retorno + ranking, por ticker/dia)
         + tabela `poc_b3_modernizacao.gold.indice_proxy` (índice-proxy + dispersão, por dia)

In [0]:
# imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# le a tabela silver (dados limpos)
df_silver = spark.table("poc_b3_modernizacao.silver.cotacoes")

print(f"Total de registros na Silver: {df_silver.count()}")
display(df_silver)

In [0]:
# indicador 1 - retorno diario por ticker (em percentual, 2 casas decimais)
df_retorno = df_silver.withColumn(
    "retorno_diario_pct",
    F.round(((F.col("preco_atual") - F.col("fechamento_anterior")) / F.col("fechamento_anterior")) * 100, 2)
)

display(df_retorno.select("ticker", "data_referencia", "preco_atual", "fechamento_anterior", "retorno_diario_pct"))

In [0]:
# indicador 3 - ranking de valorizacao (maior para o menor retorno do dia)
df_ranking = df_retorno.withColumn(
    "ranking_valorizacao",
    F.row_number().over(
        Window.partitionBy("data_referencia").orderBy(F.col("retorno_diario_pct").desc())
    )
)

display(df_ranking.select("ranking_valorizacao", "ticker", "data_referencia", "retorno_diario_pct").orderBy("ranking_valorizacao"))

In [0]:
# indicador 2 e 4 - indice-proxy (media) e dispersao (desvio padrao), por dia
df_indice = df_retorno.groupBy("data_referencia").agg(
    F.round(F.avg("retorno_diario_pct"), 2).alias("indice_proxy_pct"),
    F.round(F.stddev("retorno_diario_pct"), 2).alias("dispersao_pct"),
    F.count("ticker").alias("qtd_tickers")
)

display(df_indice)

In [0]:
# funcao auxiliar de merge (mesma logica usada na silver)
from delta.tables import DeltaTable

def merge_ou_cria(df, nome_tabela, colunas_chave):
    condicao = " AND ".join([f"destino.{c} = origem.{c}" for c in colunas_chave])
    if spark.catalog.tableExists(nome_tabela):
        tabela_delta = DeltaTable.forName(spark, nome_tabela)
        (tabela_delta.alias("destino")
            .merge(df.alias("origem"), condicao)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f"MERGE executado em {nome_tabela}")
    else:
        df.write.format("delta").saveAsTable(nome_tabela)
        print(f"Tabela {nome_tabela} criada pela primeira vez")

In [0]:
# grava gold via merge
merge_ou_cria(
    df_ranking.select("ticker", "data_referencia", "preco_atual", "fechamento_anterior", "retorno_diario_pct", "ranking_valorizacao"),
    "poc_b3_modernizacao.gold.indicadores_diarios",
    ["ticker", "data_referencia"]
)

merge_ou_cria(
    df_indice,
    "poc_b3_modernizacao.gold.indice_proxy",
    ["data_referencia"]
)

In [0]:
# valida gravacao na gold
print("=== gold.indicadores_diarios ===")
display(spark.table("poc_b3_modernizacao.gold.indicadores_diarios").orderBy("ranking_valorizacao"))

print("=== gold.indice_proxy ===")
display(spark.table("poc_b3_modernizacao.gold.indice_proxy"))